<a href="https://colab.research.google.com/github/quprep/quprep/blob/main/examples/03_encoders.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/quprep/quprep/v0.6.0?labpath=examples%2F03_encoders.ipynb)

# 03 — Encoders Compared

All **12 encoders** side by side:

| # | Encoder | Strategy | Qubits | Depth |
|---|---|---|---|---|
| 1 | `AngleEncoder` | Ry/Rx/Rz per qubit | d | O(1) |
| 2 | `AmplitudeEncoder` | State preparation | log₂(d) | O(d) |
| 3 | `BasisEncoder` | Threshold binarisation, X gates | d | O(1) |
| 4 | `EntangledAngleEncoder` | Rotation + CNOT layers | d | O(layers·d) |
| 5 | `IQPEncoder` | H + Rz + ZZ (Havlíček 2019) | d | O(d²·reps) |
| 6 | `ReUploadEncoder` | Repeated rotation (Pérez-Salinas 2020) | d | O(layers·d) |
| 7 | `HamiltonianEncoder` | Trotterised Z Hamiltonian | d | O(trotter·d) |
| 8 | `ZZFeatureMapEncoder` | H + Rz + ZZ (Qiskit convention) | d | O(d²·reps) |
| 9 | `PauliFeatureMapEncoder` | Configurable Pauli strings | d | O(d²·reps) |
| 10 | `RandomFourierEncoder` | RBF kernel approx (requires fit) | n_components | O(1) |
| 11 | `TensorProductEncoder` | Ry+Rz per qubit, 2 features/qubit | ⌈d/2⌉ | O(1) |
| 12 | `QAOAProblemEncoder` | QAOA cost Hamiltonian feature map | d | O(p) |

In [ ]:
import numpy as np

import quprep as qd
from quprep.encode.pauli_feature_map import PauliFeatureMapEncoder
from quprep.encode.qaoa_problem import QAOAProblemEncoder
from quprep.encode.random_fourier import RandomFourierEncoder
from quprep.encode.tensor_product import TensorProductEncoder
from quprep.encode.zz_feature_map import ZZFeatureMapEncoder

exporter = qd.QASMExporter()
feature_vector = np.array([0.1, 0.9, 0.4, 0.6])
x3 = np.array([0.5, 1.2, 0.75])

## 1. AngleEncoder

Each feature maps to a rotation angle on one qubit. No entanglement.

In [ ]:
angle_input = feature_vector * np.pi
enc_angle = qd.AngleEncoder(rotation='ry').encode(angle_input)
print(f"Metadata : {enc_angle.metadata}")
print(exporter.export(enc_angle))

## 2. AmplitudeEncoder

$|\psi(x)\rangle = \sum_i x_i |i\rangle$ where $\|x\|_2 = 1$. Uses $\log_2 d$ qubits.

In [ ]:
unit_vec = feature_vector / np.linalg.norm(feature_vector)
enc_amp = qd.AmplitudeEncoder().encode(unit_vec)
print(f"n_qubits : {enc_amp.metadata['n_qubits']}  (log₂(4) = 2)")
print(f"padded   : {enc_amp.metadata['padded']}")
print("(QASM not supported — use QiskitExporter)")

## 3. BasisEncoder

Binarizes features: value ≥ threshold → X gate (|1⟩), else |0⟩.

In [ ]:
enc_basis = qd.BasisEncoder(threshold=0.5).encode(feature_vector)
print(f"Parameters : {enc_basis.parameters}  (binary)")
print(exporter.export(enc_basis))

## 4. EntangledAngleEncoder

Rotation layer + CNOT entangling layer, repeated `layers` times. Topologies: linear, circular, full.

In [ ]:
enc_ent = qd.EntangledAngleEncoder(
    rotation='ry', layers=2, entanglement='circular'
).encode(x3 * np.pi)
print(f"n_qubits   : {enc_ent.metadata['n_qubits']}")
print(f"cnot_pairs : {enc_ent.metadata['cnot_pairs']}")
print(f"depth      : {enc_ent.metadata['depth']}")
print(exporter.export(enc_ent))

## 5. IQPEncoder

Havlíček et al. 2019. Hadamard + Rz + ZZ interactions. Best for kernel methods.

In [ ]:
enc_iqp = qd.IQPEncoder(reps=1).encode(x3 * np.pi)
print(f"n_qubits : {enc_iqp.metadata['n_qubits']}")
print(f"n_pairs  : {enc_iqp.metadata['n_pairs']}")
print(f"depth    : {enc_iqp.metadata['depth']}")
print(exporter.export(enc_iqp))

## 6. ReUploadEncoder

Pérez-Salinas et al. 2020. Same rotation layer repeated L times. Universal approximator with enough layers.

In [ ]:
enc_ru = qd.ReUploadEncoder(layers=3, rotation='ry').encode(x3 * np.pi)
print(f"n_qubits : {enc_ru.metadata['n_qubits']}")
print(f"layers   : {enc_ru.metadata['layers']}")
print(f"depth    : {enc_ru.metadata['depth']}")
print(exporter.export(enc_ru))

## 7. HamiltonianEncoder

Trotterised Z Hamiltonian: $e^{-iH(x)T}$. Best for physics simulation / VQE.

In [ ]:
enc_ham = qd.HamiltonianEncoder(evolution_time=1.0, trotter_steps=4).encode(x3)
print(f"n_qubits      : {enc_ham.metadata['n_qubits']}")
print(f"trotter_steps : {enc_ham.metadata['trotter_steps']}")
print(f"depth         : {enc_ham.metadata['depth']}")
print(exporter.export(enc_ham))

## 8. ZZFeatureMapEncoder

Havlíček 2019 Qiskit convention. H + Rz(2(π−xᵢ)) + pairwise ZZ interactions.

In [ ]:
enc_zz = ZZFeatureMapEncoder(reps=1).encode(x3 * np.pi)
print(f"n_qubits : {enc_zz.metadata['n_qubits']}")
print(f"depth    : {enc_zz.metadata['depth']}")
print(exporter.export(enc_zz))

## 9. PauliFeatureMapEncoder

Generalised Pauli strings. `["Z", "ZZ"]` is equivalent to ZZFeatureMap.

In [ ]:
enc_pf = PauliFeatureMapEncoder(paulis=['Z', 'ZZ'], reps=1).encode(x3 * np.pi)
print(f"n_qubits : {enc_pf.metadata['n_qubits']}")
print(f"depth    : {enc_pf.metadata['depth']}")
print(exporter.export(enc_pf))

## 10. RandomFourierEncoder

RBF kernel approximation via Bochner's theorem. Must call `fit()` before `encode()`.

In [ ]:
data_batch = np.random.default_rng(0).uniform(0, 1, size=(20, 3))
enc_rf_obj = RandomFourierEncoder(n_components=6, random_state=0)
enc_rf_obj.fit(data_batch)
enc_rf = enc_rf_obj.encode(x3)
print(f"n_qubits : {enc_rf.metadata['n_qubits']}  (= n_components)")
print(f"depth    : {enc_rf.metadata['depth']}")
print(exporter.export(enc_rf))

## 11. TensorProductEncoder

Ry + Rz per qubit — full Bloch sphere, 2 features per qubit. No entanglement.

In [ ]:
enc_tp = TensorProductEncoder().encode(x3 * np.pi)
print(f"n_qubits : {enc_tp.metadata['n_qubits']}  (ceil(3/2) = 2)")
print(f"depth    : {enc_tp.metadata['depth']}")
print(exporter.export(enc_tp))

## 12. QAOAProblemEncoder

QAOA-inspired feature map. Features as cost Hamiltonian parameters. H + RZ(2γxᵢ) + CNOT-RZ(2γxᵢxⱼ)-CNOT + RX(2β).

In [ ]:
enc_qaoa = QAOAProblemEncoder(p=1, connectivity='linear').encode(x3 * np.pi - np.pi / 2)
print(f"n_qubits : {enc_qaoa.metadata['n_qubits']}")
print(f"n_pairs  : {enc_qaoa.metadata['n_pairs']}")
print(f"depth    : {enc_qaoa.metadata['depth']}")
print(exporter.export(enc_qaoa))

## Summary — `prepare()` one-liner

All stateless encoders work directly with `prepare()`.

In [ ]:
data = np.random.default_rng(0).uniform(0, 1, size=(10, 3))
for enc_name in ('angle', 'basis', 'iqp', 'reupload', 'hamiltonian',
                  'zz_feature_map', 'tensor_product', 'qaoa_problem'):
    result = qd.prepare(data, encoding=enc_name, framework='qasm')
    meta = result.encoded[0].metadata
    print(f"  {enc_name:20s}  qubits={meta['n_qubits']}  depth={meta['depth']}")